# 🕵️ Dia 22 — O ciclo OSINT, parte 4: analisando um grafo de vínculos e relatando

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 22* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-22.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A Marta quer "uma página, até o meio-dia". Você aponta para o `203.0.113.77`: *"É ele, tudo passa por ele."* Rafa cobre o nó com o polegar: nada cai. Depois cobre um nó pequeno, `G-ISCA7K2Q9P`, e o `coopserra-azul.example` fica sozinho no canto.

Neste notebook você vai:

1. **Calcular grau e intermediação** (algoritmo de Brandes) só com a biblioteca padrão.
2. **Fazer o teste do polegar** em código.
3. **Jogar "hub, ponte ou nenhum dos dois?"**.
4. **Confirmar a ponte** nas capturas e gerar a seção Análise e Relatório, com checagem automática de confiança.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. Grau e intermediação

**Grau** = quantas conexões o nó tem (destaca hubs). **Intermediação** = em quantos caminhos mais curtos entre outros nós ele está (destaca pontes). Primeiro, o grafo ilustrativo do original (seção Conceito do README).

In [ ]:
from collections import deque, defaultdict

def montar(arestas):
    """Lista de arestas (a, b, ...) → dicionário de adjacência {nó: {vizinhos}}."""
    adj = defaultdict(set)
    for a, b, *_resto in arestas:
        adj[a].add(b)                    # não direcionado: a liga b...
        adj[b].add(a)                    # ...e b liga a
    return dict(adj)

def intermediacao(adj):
    """Brandes (2001): centralidade de intermediação, sem normalizar."""
    bc = dict.fromkeys(adj, 0.0)
    for s in adj:                                         # um BFS a partir de cada nó
        pilha, pred = [], {v: [] for v in adj}
        sigma, dist = dict.fromkeys(adj, 0), dict.fromkeys(adj, -1)
        sigma[s], dist[s], fila = 1, 0, deque([s])
        while fila:
            v = fila.popleft(); pilha.append(v)
            for w in adj[v]:
                if dist[w] < 0:                           # primeira vez que vemos w
                    dist[w] = dist[v] + 1; fila.append(w)
                if dist[w] == dist[v] + 1:                # w está um passo depois de v
                    sigma[w] += sigma[v]; pred[w].append(v)
        delta = dict.fromkeys(adj, 0.0)
        while pilha:                                      # volta acumulando a "dívida" de cada nó
            w = pilha.pop()
            for v in pred[w]:
                delta[v] += sigma[v] / sigma[w] * (1 + delta[w])
            if w != s:
                bc[w] += delta[w]
    return {v: b / 2 for v, b in bc.items()}             # /2 porque cada par foi contado duas vezes

def normalizar(bc):
    """Escala 0–1, como o Gephi mostra quando normalizado."""
    n = len(bc)
    fator = (n - 1) * (n - 2) / 2 if n > 2 else 1
    return {v: b / fator for v, b in bc.items()}

def tabela(adj, top=6):
    bc = normalizar(intermediacao(adj))
    print(f"{'Nó':<34}{'Grau':>6}{'Intermed.':>11}")
    for v in sorted(adj, key=lambda x: -bc[x])[:top]:
        print(f"{v:<34}{len(adj[v]):>6}{bc[v]:>11.2f}")
    return bc

# Grafo ilustrativo do original (diagrama do Conceito)
ILUSTRATIVO = [("example.org", h) for h in ["www.example.org", "news.example.org", "jobs.example.org",
                "library.example.org", "portal.example.org"]]
ILUSTRATIVO += [(h, "192.0.2.80") for h in ["www.example.org", "news.example.org", "jobs.example.org", "library.example.org"]]
ILUSTRATIVO += [("portal.example.org", "192.0.2.10"), ("192.0.2.10", "AS64510"),
                ("www.example.org", "G-ABC123XYZ"), ("G-ABC123XYZ", "example-events.org"),
                ("G-ABC123XYZ", "www.example-events.org"), ("example-events.org", "www.example-events.org"),
                ("example-events.org", "198.51.100.140"), ("www.example-events.org", "198.51.100.140")]
adj_ilus = montar(ILUSTRATIVO)
_ = tabela(adj_ilus)
print("\nRepare: o hub da CDN (192.0.2.80) tem grau 4 e a intermediação mais baixa da tabela. A ponte G-ABC123XYZ está no topo.")

### Agora o grafo do caso

Mesmas arestas de [`caso-continuo/dados/dia-22/isca-grafo.csv`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/tree/main/caso-continuo/dados/dia-22), com rótulo e fonte.

In [ ]:
# (origem, destino, rótulo, fonte)
ARESTAS_ISCA = [
 [
  "serraazul-pix.example",
  "www.serraazul-pix.example",
  "nome DNS",
  "Maltego, tabela D21 linha 2"
 ],
 [
  "serraazul-pix.example",
  "admin.serraazul-pix.example",
  "nome DNS",
  "Maltego, tabela D21 linha 3"
 ],
 [
  "serraazul-pix.example",
  "cdn.serraazul-pix.example",
  "nome DNS",
  "Maltego, tabela D21 linha 4"
 ],
 [
  "serraazul-pix.example",
  "ns1.dnsbarato.example",
  "NS",
  "Maltego, tabela D21 linha 1"
 ],
 [
  "www.serraazul-pix.example",
  "203.0.113.77",
  "resolve para",
  "tabela D21 linha 6"
 ],
 [
  "admin.serraazul-pix.example",
  "203.0.113.77",
  "resolve para",
  "tabela D21 linha 7"
 ],
 [
  "cdn.serraazul-pix.example",
  "192.0.2.80",
  "resolve para",
  "tabela D21 linha 8"
 ],
 [
  "ns1.dnsbarato.example",
  "198.51.100.53",
  "resolve para",
  "tabela D21 linha 5"
 ],
 [
  "203.0.113.77",
  "AS64501",
  "bloco/AS",
  "tabela D21 linhas 12 e 17"
 ],
 [
  "192.0.2.80",
  "AS64500",
  "bloco/AS",
  "tabela D21 linhas 13 e 18"
 ],
 [
  "198.51.100.53",
  "AS64502",
  "bloco/AS",
  "tabela D21 linhas 14 e 19"
 ],
 [
  "serraazul-seguranca.example",
  "203.0.113.77",
  "resolve para",
  "tabela D21 linha 9"
 ],
 [
  "serraazul-seguranca.example",
  "ns1.dnsbarato.example",
  "NS",
  "dig NS, registro ISCA linha 31"
 ],
 [
  "coopserra-azul.example",
  "198.51.100.140",
  "resolve para",
  "tabela D21 linha 10"
 ],
 [
  "198.51.100.140",
  "AS64503",
  "bloco/AS",
  "tabela D21 linhas 15 e 20"
 ],
 [
  "www.serraazul-pix.example",
  "G-ISCA7K2Q9P",
  "mesmo ID de analytics",
  "captura, registro ISCA linha 32"
 ],
 [
  "coopserra-azul.example",
  "G-ISCA7K2Q9P",
  "mesmo ID de analytics",
  "captura, registro ISCA linha 33"
 ],
 [
  "serraazull.example",
  "198.51.100.200",
  "resolve para",
  "tabela D21 linha 11"
 ],
 [
  "198.51.100.200",
  "AS64504",
  "bloco/AS",
  "tabela D21 linhas 16 e 21"
 ]
]
adj = montar(ARESTAS_ISCA)
bc_isca = tabela(adj, top=8)
print("\nA semente (serraazul-pix.example) está alta 'por construção': todo salto começou nela. Não é achado.")

## 2. 👍 O teste do polegar

Remover um nó e contar em quantos pedaços (componentes conexos) o grafo fica. Se o número de pedaços aumenta, o nó era uma ponte (em teoria dos grafos: *ponto de articulação*).

In [ ]:
def componentes(adj, sem=None):
    """Lista de componentes conexos, ignorando o nó 'sem' (o polegar)."""
    vistos, partes = set(), []
    for inicio in adj:
        if inicio == sem or inicio in vistos:
            continue
        parte, fila = set(), deque([inicio])
        while fila:
            v = fila.popleft()
            if v in vistos or v == sem:
                continue
            vistos.add(v); parte.add(v)
            fila.extend(adj[v] - vistos)
        partes.append(parte)
    return partes

base = len(componentes(adj))
print(f"Sem polegar: {base} pedaço(s)  ← serraazull.example já está solto; isso é resultado honesto\n")
for polegar in ["203.0.113.77", "ns1.dnsbarato.example", "G-ISCA7K2Q9P", "www.serraazul-pix.example"]:
    partes = componentes(adj, sem=polegar)
    # Só interessa o pedaço que perdeu o caminho até a semente E contém um domínio/host.
    # Se só cai a "folha" do próprio nó (o AS de um IP, por exemplo), não é ponte de verdade.
    soltos = [sorted(p) for p in partes
              if "serraazul-pix.example" not in p and "serraazull.example" not in p
              and any(v.endswith(".example") for v in p)]
    folhas = [sorted(p) for p in partes
              if "serraazul-pix.example" not in p and "serraazull.example" not in p and p not in map(set, soltos)]
    print(f"Polegar em {polegar:<28} → {len(partes)} pedaços")
    if soltos:
        print(f"   🚨 domínio isolado do golpe: {soltos}")
    else:
        print(f"   nenhum domínio perdeu o caminho (só caiu a folha {folhas}) → não é ponte que importe")

> Repare que `www.serraazul-pix.example` também separa coisas: ele é o nó onde o ID de *analytics* foi visto. Faz sentido: a ponte real é **o par** página + ID. No relatório, você cita a captura da página, não só o ID.

## 3. 🎮 Hub, ponte ou nenhum dos dois?

1 = hub (muitas conexões, não prova propriedade) · 2 = ponte (une partes que, sem ele, se separam) · 3 = nenhum dos dois

In [ ]:
QUIZ = [
    ("203.0.113.77", "1", "Hospedagem compartilhada; pelo menos 24 vizinhos. Grau alto, intermediação baixa."),
    ("G-ISCA7K2Q9P", "2", "Única ligação de coopserra-azul.example com o resto. Confirmar na fonte primária."),
    ("ns1.dnsbarato.example", "1", "DNS comercial barato: serve muita gente."),
    ("AS64504", "3", "Folha de um componente solto (serraazull.example). Nem hub, nem ponte."),
    ("192.0.2.80", "3", "No grafo filtrado do caso só tem o cdn. pendurado; num grafo maior seria hub de CDN."),
]
MODO_JOGO = True
acertos = 0
for no_, certo, porque in QUIZ:
    print(f"\n{no_}  (grau {len(adj[no_])}, intermed. {bc_isca[no_]:.2f})")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3"):
            r = input("   1 hub / 2 ponte / 3 nenhum: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end="")
    print(f"   {['hub', 'ponte', 'nenhum'][int(certo) - 1]}: {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 4. Confirmar a ponte e escrever a página da Marta

A ponte só entra no relatório depois de: (a) conferir o hash das capturas; (b) achar o ID no código-fonte das duas.

In [ ]:
import hashlib, re
from pathlib import Path

CAPTURAS = {
 "captura-serraazul-pix.html": "<html><head><title>Serra Azul - Regularize sua chave Pix</title>\n<script async src=\"https://www.googletagmanager.com/gtag/js?id=G-ISCA7K2Q9P\"></script>\n<script>gtag('config','G-ISCA7K2Q9P');</script></head>\n<body><!-- IscaKit v2 --><p>Sua chave Pix foi bloqueada por seguranca.</p></body></html>\n",
 "captura-coopserra-azul.html": "<html><head><title>CoopSerra Azul - Atualizacao cadastral</title>\n<script async src=\"https://www.googletagmanager.com/gtag/js?id=G-ISCA7K2Q9P\"></script>\n<script>gtag('config','G-ISCA7K2Q9P');</script></head>\n<body><!-- IscaKit v2 --><p>Atualize seus dados para evitar o bloqueio da conta.</p></body></html>\n"
}
# Recria as capturas e o arquivo de hashes (idênticos aos de caso-continuo/dados/dia-22/)
hashes = {}
for nome, conteudo in CAPTURAS.items():
    Path(nome).write_text(conteudo, encoding="utf-8")
    hashes[nome] = hashlib.sha256(conteudo.encode("utf-8")).hexdigest()

ID = "G-ISCA7K2Q9P"
padrao_ga = re.compile(r"G-[A-Z0-9]{6,12}")          # formato de ID de medição do GA4
for nome in CAPTURAS:
    dados = Path(nome).read_bytes()
    ok_hash = hashlib.sha256(dados).hexdigest() == hashes[nome]     # (a) integridade
    ids = set(padrao_ga.findall(dados.decode("utf-8")))           # (b) o que está no código
    kit = "IscaKit v2" in dados.decode("utf-8")
    print(f"{nome}: hash {'OK' if ok_hash else 'FALHOU'} · IDs {sorted(ids)} · comentário do kit: {kit}")
print(f"\nPonte {ID} confirmada nas duas capturas." )

In [ ]:
#@title Seção Análise e Relatório { display-mode: "form" }
linha_cap_pix = 32  #@param {type:"integer"}
linha_cap_coop = 33  #@param {type:"integer"}
confianca_coopserra = "confirmado"  #@param ["confirmado", "provável", "contestado"]
confianca_seguranca = "provável"  #@param ["confirmado", "provável", "contestado"]

# A tabela de Análise vem PRIMEIRO. O relatório copia a palavra dela.
ANALISE = [
    ("coopserra-azul.example é da mesma operação que serraazul-pix.example", confianca_coopserra,
     f"{linha_cap_pix}, {linha_cap_coop}"),
    ("serraazul-seguranca.example é da mesma operação", confianca_seguranca, "tabela D21 linha 9; registro linha 31"),
    ("203.0.113.77 é hub de hospedagem compartilhada, não prova propriedade", "confirmado", "D20 (412 co-hospedados); D21 reverse IP (pelo menos 24)"),
    ("serraazull.example não tem ligação com a campanha", "provável", "tabela D21 linhas 11, 16, 21"),
]

relatorio = f"""coopserra-azul.example é **{confianca_coopserra}** como parte da mesma operação que serraazul-pix.example: as duas páginas carregam o mesmo ID de medição G-ISCA7K2Q9P (capturas com SHA-256 nas linhas {linha_cap_pix} e {linha_cap_coop} do registro) e o mesmo comentário de versão do IscaKit. Os dois sites não compartilham hospedagem, DNS nem AS; a infraestrutura não faz parte desta conclusão.
serraazul-seguranca.example é **{confianca_seguranca}**: as únicas ligações são o IP 203.0.113.77, uma hospedagem compartilhada com pelo menos 24 outros sites, e o NS ns1.dnsbarato.example; dois hubs não somam uma ponte.
O IP 203.0.113.77 é um hub **confirmado** de hospedagem compartilhada e não indica propriedade comum.
serraazull.example não tem ligação com a campanha (**provável**): outro host, outro AS, página de domínio à venda.
O que mudaria estes achados: uma segunda fonte independente para o ID (ex.: outra captura de terceiro) ou um registro RDAP comum (Dia 29)."""

# Checagem automática do Checkpoint: toda palavra em negrito no relatório existe na tabela, na mesma ordem.
palavras_rel = re.findall(r"\*\*(confirmado|provável|contestado)\*\*", relatorio)
palavras_tab = [c for _a, c, _f in ANALISE]
print("| Achado | Confiança | Fontes |\n|---|---|---|")
for a, c, f in ANALISE:
    print(f"| {a} | {c} | {f} |")
print("\n" + relatorio)
assert palavras_rel == palavras_tab, f"Relatório {palavras_rel} ≠ tabela {palavras_tab}"
print("\n✅ Toda palavra de confiança do relatório bate com a tabela de Análise.")
Path("notas").mkdir(exist_ok=True)
Path("notas/ISCA-analise-relatorio.md").write_text(
    "## Análise\n| Achado | Confiança | Fontes |\n|---|---|---|\n"
    + "".join(f"| {a} | {c} | {f} |\n" for a, c, f in ANALISE)
    + "\n## Relatório\n" + relatorio + "\n", encoding="utf-8")
print("💾 notas/ISCA-analise-relatorio.md")

### 🧠 Camada 8

O nó grande no meio do desenho é **saliente**; o pequeno no canto prova alguma coisa. O teste do polegar é pensamento contrafactual deliberado. E a palavra de confiança só significa algo se a tabela vier antes do texto.

**Discussão para as notas:** o ID de *analytics* e o comentário do kit vieram do **mesmo** código-fonte. São duas fontes independentes? Se você acha que não, mude `confianca_coopserra` para "provável" no formulário, rode de novo e veja a checagem continuar passando: o que importa é a coerência entre tabela e relatório, e a justificativa escrita.

## ✅ Antes de fechar o Dia 22

- [ ] Você sabe dizer por que o `203.0.113.77` tem grau alto e intermediação baixa
- [ ] Teste do polegar feito; a ponte e o componente solto identificados
- [ ] Ponte confirmada (hash + ID nas duas capturas)
- [ ] Análise e Relatório gerados, checagem ✅
- [ ] Discussão sobre independência das fontes escrita

**Amanhã (Dia 23):** operadores de busca. O buscador já indexou o que o golpe esqueceu que publicou, e o que a cooperativa esqueceu também.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>